# 03 · Ask questions about a screenshot

**You will:** send an image as evidence, reuse the three decision types, locate a banner with Box, and handle an absent target.

We extend support triage with screenshots from a fictional application. These original fixtures ship with the repository. They demonstrate mechanics; they are not a realistic vision benchmark.

Images and Box are experimental TypeSafe features. Both Jev 1.13 and 1.14 accept image input with the opt-in header; Box requires 1.14. Use the explicit model version and header below.

In [ ]:
# Find the course folder and load API settings without printing secret keys.
from pathlib import Path
import sys
import json
from IPython.display import display
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'course_utils').is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open Jupyter inside the course folder.')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from course_utils.config import load_settings, require_key
settings = load_settings(ROOT)
print('Python:', sys.version.split()[0], '| Mode:', settings.mode)
print('Jev key configured:', bool(settings.typesafe_api_key))
print('OpenAI key configured:', bool(settings.openai_api_key))

In [ ]:
# Import the decision types and configure live calls or authentic replay.
from contextlib import nullcontext
from typesafe_sdk import TypeSafeClient, Choice, Noul, RetryPolicy
from course_utils.records import run_case
from course_utils.evaluation import route_team

RUN_DIR = ROOT / 'work/runs'
can_jev = settings.mode == 'recorded' or bool(settings.typesafe_api_key)
can_openai = settings.mode == 'recorded' or bool(settings.openai_api_key)


def jev_client():
    if settings.mode == 'recorded':
        return nullcontext(None)
    return TypeSafeClient(
        api_key=require_key(settings, 'typesafe'),
        retry=RetryPolicy(max_retries=1),
        timeout=30,
    )

In [ ]:
# Load the screenshot list and enable image and Box questions.
import hashlib
from typesafe_sdk import Image as JevImage, Box, Score
from PIL import Image as PILImage
from course_utils.plots import draw_box

image_cases = []
for line in (ROOT / 'data/image_cases.jsonl').read_text().splitlines():
    image_cases.append(json.loads(line))
IMAGE_HEADERS = {'TypeSafe-Danger-Zone-Images': 'true'}

In [ ]:
# Define the category, severity, presence, and location questions.
image_questions = {
    'category': Choice(
        instructions='What kind of error or warning is visible in the screenshot?',
        criteria={
            'payment': 'A visible payment or checkout error.',
            'login': 'A visible sign-in or account access error.',
            'service': 'A visible service or connection error/warning.',
            'none': 'No visible error or warning.',
        },
    ),
    'severity': Score(
        instructions='How much work does the visible issue block?',
        criteria=[
            'No error or warning.',
            'Warning, but work can continue.',
            'One specific task is blocked.',
            'The whole service is blocked.',
        ],
    ),
    'error_present': Noul(
        instructions='Is an error or warning banner visible? Success and empty-state notices are not errors.',
    ),
    'banner': Box(instructions='Locate the error or warning banner in the screenshot.'),
}

In [ ]:
# Define a recorded image request; each call sends the required opt-in header.
def ask_image(example_id, path, questions):
    if not can_jev:
        print('Configure TYPESAFE_API_KEY or select authentic recordings.')
        return None
    config = {
        'image_sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
        'state_field': 'screenshot',
        'questions': {name: question.model_dump(mode='json') for name, question in questions.items()},
        'headers': IMAGE_HEADERS,
        'max_retries': 1,
        'timeout': 30,
    }
    with jev_client() as client:
        def request():
            return client.system_one(
                model=settings.jev_new,
                state={'screenshot': JevImage(path)},
                questions=questions,
                extra_headers=IMAGE_HEADERS,
            )
        return run_case(
            request, provider='typesafe', endpoint='systemone', model=settings.jev_new,
            example_id=example_id, config=config, question_names=list(questions),
            mode=settings.mode, directory=RUN_DIR,
        )

## Select one screenshot
Start with the payment screen. Then try `image-success` or `image-empty` to inspect an absent target. These are synthetic examples, not a benchmark.

In [ ]:
# Choose a screenshot and display the exact image sent to the model.
example_id = 'image-payment'
image_case = next(case for case in image_cases if case['id'] == example_id)
image_path = ROOT / 'data' / image_case['image_path']
print('State analysed:', image_path.name)
display(PILImage.open(image_path))

## Same evidence, different answer shapes

Category is unordered, severity uses defined levels, and presence is yes/no. None requires a generated paragraph. Box answers a different question: **where** is the target?

We pass a local PNG through `JevImage(path)` and explicitly enable images. The helper above contains the actual SDK call; `run_case` only records/times that call or loads an explicitly requested authentic recording.

In [ ]:
# Ask the four image questions and show the returned answers.
image_result = ask_image(example_id, image_path, image_questions)
if image_result is not None:
    print('State analysed:', image_path.name)
    print('Status:', image_result.status, '| Error:', image_result.error)
    display(image_result.answers)

## A box is not evidence of presence

Box always returns coordinates for a valid answer, including when the target is absent. Gate it using a separate presence judgment. The 0.8 threshold here is illustrative; six synthetic images are not enough to validate production behavior. Box confidence and presence probability are separate signals.

Coordinates are fractions of image width/height. Overlay on the **same image dimensions** sent to the model. Our fixtures are 1000×640 PNGs. For other inputs, follow the documented size limits, correct orientation before upload, and inspect readability after resizing.

In [ ]:
# Draw the predicted banner box only when its presence probability is high enough.
if image_result is not None:
    presence_probability = image_result.answers.get('error_present', {}).get('noul', 0)
    coordinates = image_result.answers.get('banner', {}).get('box')
    with PILImage.open(image_path) as image:
        display(draw_box(image, coordinates, present_probability=presence_probability, threshold=0.8))

## Exercise: locate a button
Ask whether a Continue button is visible, then locate it. Suppress the box when presence is uncertain. This example performs no clicks.

In [ ]:
# Locate the Continue button using the same image helper.
button_questions = {
    'present': Noul(instructions='Is a button labelled Continue visible in the screenshot?'),
    'button': Box(instructions='Locate the button labelled Continue in the screenshot.'),
}
button_result = ask_image('button-' + example_id, image_path, button_questions)
if button_result is not None:
    print('State analysed:', image_path.name)
    print('Status:', button_result.status, '| Error:', button_result.error)
    display(button_result.answers)
    presence_probability = button_result.answers.get('present', {}).get('noul', 0)
    coordinates = button_result.answers.get('button', {}).get('box')
    with PILImage.open(image_path) as image:
        display(draw_box(image, coordinates, present_probability=presence_probability, threshold=0.8))

Sources: [image inputs and preparation](https://docs.typesafe.ai/images), [Box and absent targets](https://docs.typesafe.ai/primitives/box). Image decisions will also be compared with OpenAI in notebook 04.